# 🎨 Gambar AI Willo (Kaggle, gratis)
Langkah sekali saja:
1. Kanan: **Settings → Accelerator → GPU T4 x2** (atau P100).
2. **Settings → Internet → On** (akun Kaggle harus sudah verifikasi nomor HP).
3. (Opsional, supaya hasil masuk ke @Willo_video_bot) **Add-ons → Secrets → Add**: nama `TG_VIDEO`, isi token bot. Centang (attach) ke notebook ini.

Lalu tekan **Run All**. Pertama kali butuh ±5–10 menit untuk unduh model. Gambar berikutnya cukup ubah `PROMPT` di sel 2 lalu jalankan sel 2 dan 4 saja.

Model: **Z-Image Turbo** (bawaan, cepat, Apache 2.0, boleh untuk jualan) atau **Qwen-Image** (resmi, Apache 2.0, lebih lambat).

In [ ]:
# Sel 1: pasang pustaka (sekali per sesi)
!pip install -q -U git+https://github.com/huggingface/diffusers transformers accelerate gguf sentencepiece


In [ ]:
# Sel 2: ATUR DI SINI
MODEL   = "z-image"        # "z-image" (cepat) atau "qwen-image" (lebih lambat)
PROMPT  = "foto kucing oranye berbulu duduk di ambang jendela kayu, cahaya pagi lembut, sangat detail"
LEBAR, TINGGI = 1024, 1024 # untuk video vertikal pakai 768, 1344
JUMLAH  = 2                # berapa gambar
KIRIM_TELEGRAM = True      # kirim ke @Willo_video_bot (butuh Secret TG_VIDEO)


In [ ]:
# Sel 3: muat model (sekali per sesi; jalankan ulang hanya kalau MODEL diganti)
import torch, os, glob
from huggingface_hub import HfApi, hf_hub_download
DT = torch.bfloat16
if MODEL == "z-image":
    from diffusers import ZImagePipeline
    pipe = ZImagePipeline.from_pretrained("Tongyi-MAI/Z-Image-Turbo", torch_dtype=DT)
    LANGKAH, CFG = 9, 0.0
else:
    from diffusers import QwenImagePipeline, QwenImageTransformer2DModel, GGUFQuantizationConfig
    berkas = [f for f in HfApi().list_repo_files("city96/Qwen-Image-gguf") if f.endswith(".gguf") and "Q4_K_M" in f][0]
    tr = QwenImageTransformer2DModel.from_single_file(
        hf_hub_download("city96/Qwen-Image-gguf", berkas),
        quantization_config=GGUFQuantizationConfig(compute_dtype=DT), torch_dtype=DT,
        config="Qwen/Qwen-Image", subfolder="transformer")
    pipe = QwenImagePipeline.from_pretrained("Qwen/Qwen-Image", transformer=tr, torch_dtype=DT)
    LANGKAH, CFG = 20, 4.0
pipe.enable_model_cpu_offload()
print("Model siap:", MODEL)


In [ ]:
# Sel 4: buat gambar
import time, requests
from IPython.display import display
token = None
if KIRIM_TELEGRAM:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("TG_VIDEO")
    except Exception as e:
        print("Secret TG_VIDEO belum dipasang, gambar tidak dikirim ke bot.")
os.makedirs("/kaggle/working/hasil", exist_ok=True)
for i in range(JUMLAH):
    t = time.time()
    arg = dict(prompt=PROMPT, width=LEBAR, height=TINGGI, num_inference_steps=LANGKAH,
               generator=torch.Generator("cpu").manual_seed(int(t) + i))
    if MODEL == "z-image": arg["guidance_scale"] = CFG
    else: arg.update(true_cfg_scale=CFG, negative_prompt=" ")
    img = pipe(**arg).images[0]
    detik = time.time() - t
    path = f"/kaggle/working/hasil/{MODEL}-{int(t)}-{i+1}.png"
    img.save(path); display(img.resize((img.width // 2, img.height // 2)))
    print(f"Gambar {i+1}: {detik:.0f} detik → {path}")
    if token:
        with open(path, "rb") as f:
            r = requests.post(f"https://api.telegram.org/bot{token}/sendDocument",
                              data={"chat_id": "5667928446", "caption": f"{MODEL} {LEBAR}x{TINGGI}, {detik:.0f} dtk\n{PROMPT[:900]}"},
                              files={"document": f})
        print("Terkirim ke bot" if r.ok else f"Gagal kirim ke bot: {r.text[:200]}")
